In [1]:
%%capture
!pip install --force-reinstall nxd.core nxd.data_product[notebook]

In [2]:
%%capture
!pip install --force-reinstall altair snowflake-connector-python[secure-local-storage,pandas]

In [3]:
from nxd.data_product.client import create_client
import nxd.data_product.context as ctx
import altair as alt
from snowflake.connector import connect

Loading nxd.data_product v0.1.4


# Term Deposit Rates

In [ ]:
nxd_client = create_client(hostname="dp.demo.trynxd.com", user="luke.simplemachines@nextdata.com")
data_product = nxd_client.data_product(data_product="product-competitiveness-demo")
snowflake = data_product.get_outputs("snowflake", ctx.Snowflake)

conn = connect(
    user=snowflake.user,
    password=snowflake.password,
    account=snowflake.account,
    warehouse=snowflake.warehouse,
    database=snowflake.database,
    schema=snowflake.schema,
)
cursor = conn.cursor()

## Interest Paid At Maturity

In [5]:
maturity_rates = cursor.execute(r"""
     SELECT
        min_term,
        bank,
        max(maturity_rate) as maturity_rate
    FROM term_deposits
    WHERE min_term <= 12 and maturity_rate is not null
    GROUP BY
        bank,
        min_term
    ORDER BY
        min_term""",
).fetch_pandas_all()

alt.Chart(maturity_rates).mark_point(filled=True, size=100).encode(
    x=alt.X("MIN_TERM:Q", axis=alt.Axis(tickCount=12)).scale(domain=(0, 13)).title("Minimum Term (months)"),
    y=alt.Y("MATURITY_RATE").title("Maturity Rate (% p.a)"),
    color=alt.Color("BANK", scale=alt.Scale(domain=["Westpac", "ANZ", "Macquarie"], range=["red", "blue", "black"])),
    tooltip=["MIN_TERM", "MATURITY_RATE"]
).properties(
    height=600,
    width=1000,
).interactive()


alt.Chart(...)

## Interest Paid Annually

In [6]:
annual_rates = cursor.execute(r"""
    SELECT
        min_term,
        bank,
        max(annual_rate) as annual_rate
    FROM term_deposits
    WHERE min_term >= 12 and min_term < 60 and annual_rate is not null
    GROUP BY
        bank,
        min_term
    ORDER BY
        min_term   
""").fetch_pandas_all()

alt.Chart(annual_rates).mark_point(filled=True, size=100).encode(
    x=alt.X("MIN_TERM:Q").scale(domain=(23, 60)).title("Minimum Term (months)"),
    y=alt.Y("ANNUAL_RATE").title("Annual Rate (% p.a)"),
    color=alt.Color("BANK", scale=alt.Scale(domain=["Westpac", "ANZ", "Macquarie"], range=["red", "blue", "black"])),
    tooltip=["MIN_TERM", "ANNUAL_RATE"]
).properties(
    height=600,
    width=1000,
).interactive()


alt.Chart(...)

## Interest Paid Monthly

In [7]:
monthly_rates = cursor.execute(r"""
     SELECT
        min_term,
        bank,
        max(monthly_rate) as monthly_rate
    FROM term_deposits
    WHERE min_term >= 2 and min_term <= 12 and maturity_rate is not null
    GROUP BY
        bank,
        min_term
    ORDER BY
        min_term    
""").fetch_pandas_all()

alt.Chart(monthly_rates).mark_point(filled=True, size=100).encode(
    x=alt.X("MIN_TERM:Q", axis=alt.Axis(tickCount=12)).scale(domain=(1, 13)).title("Minimum Term (months)"),
    y=alt.Y("MONTHLY_RATE").title("Monthly Rate (% p.a)"),
    color=alt.Color("BANK", scale=alt.Scale(domain=["Westpac", "ANZ", "Macquarie"], range=["red", "blue", "black"])),
    tooltip=["MIN_TERM", "MONTHLY_RATE"]
).properties(
    height=600,
    width=1000,
).interactive()

alt.Chart(...)

# Interest Rates

## Fixed Home Loan (LVR <70)

In [8]:
target_lvr = 70

interest_rates = cursor.execute(f"""
    select
        bank,
        loan_term,
        min(rate) as rate
    from home_loan_rates
    where {target_lvr} > min_lvr and {target_lvr} <= max_lvr
    group by
        bank,
        loan_term
    order by
        loan_term
""").fetch_pandas_all()

alt.Chart(interest_rates).mark_point(filled=True, size=100).encode(
    x=alt.X("LOAN_TERM:Q", axis=alt.Axis(tickCount=6)).scale(domain=(0, 6)).title("Loan Term (years)"),
    y=alt.Y("RATE").title("Interest (% p.a)").scale(domain=(0, 7)).title("Interest Rate (% pa)"),
    color=alt.Color("BANK", scale=alt.Scale(domain=["Westpac", "ANZ", "Macquarie"], range=["red", "blue", "black"])),
    tooltip=["LOAN_TERM", "RATE"]
).properties(
    height=600,
    width=1000,
).interactive()

alt.Chart(...)

## Fixed Home Loan (LVR 70-80)

In [9]:
target_lvr = 80

interest_rates = cursor.execute(f"""
    select
        bank,
        loan_term,
        min(rate) as rate
    from home_loan_rates
    where {target_lvr} > min_lvr and {target_lvr} <= max_lvr
    group by
        bank,
        loan_term
    order by
        loan_term
""").fetch_pandas_all()

alt.Chart(interest_rates).mark_point(filled=True, size=100).encode(
    x=alt.X("LOAN_TERM:Q", axis=alt.Axis(tickCount=6)).scale(domain=(0, 6)).title("Loan Term (years)"),
    y=alt.Y("RATE").title("Interest (% p.a)").scale(domain=(0, 7)).title("Interest Rate (% pa)"),
    color=alt.Color("BANK", scale=alt.Scale(domain=["Westpac", "ANZ", "Macquarie"], range=["red", "blue", "black"])),
    tooltip=["LOAN_TERM", "RATE"]
).properties(
    height=600,
    width=1000,
).interactive()

alt.Chart(...)

## Fixed Home Loan (LVR 80+)

In [10]:
target_lvr = 90

interest_rates = cursor.execute(f"""
    select
        bank,
        loan_term,
        min(rate) as rate
    from home_loan_rates
    where {target_lvr} > min_lvr and {target_lvr} <= max_lvr
    group by
        bank,
        loan_term
    order by
        loan_term
""").fetch_pandas_all()

alt.Chart(interest_rates).mark_point(filled=True, size=100).encode(
    x=alt.X("LOAN_TERM:Q", axis=alt.Axis(tickCount=6)).scale(domain=(0, 6)).title("Loan Term (years)"),
    y=alt.Y("RATE").title("Interest (% p.a)").scale(domain=(0, 7)).title("Interest Rate (% pa)"),
    color=alt.Color("BANK", scale=alt.Scale(domain=["Westpac", "ANZ", "Macquarie"], range=["red", "blue", "black"])),
    tooltip=["LOAN_TERM", "RATE"]
).properties(
    height=600,
    width=1000,
).interactive()

alt.Chart(...)